# Image Classification Pipeline

This notebook initializes Google Earth Engine and prepares imagery collections for supervised/unsupervised classification.

The workflow includes:
1. Earth Engine authentication
2. Area of Interest (AOI) definition
3. Temporal filtering
4. Collection selection
5. Image preprocessing scaffold

In [1]:
# ============================================================
# 1. Earth Engine Initialization
# ============================================================
import ee, geemap, pandas as pd, numpy as np, geopandas as gpd
pd.set_option('display.float_format', '{:.4f}'.format)
np.set_printoptions(precision=4, suppress=True)

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Earth Engine is ready!")
print("EE Version:", ee.__version__)

Earth Engine is ready!
EE Version: 1.6.14


# 1. Define Area of Interest (AOI)
Here we use US Census TIGER/2018 county boundaries. STATEFP = 37 corresponds to North Carolina and then we select Mecklenburg County.


In [3]:
# ============================================================
# 1. Define Area of Interest (Mecklenburg County, NC)
# ============================================================
AOI = (
    ee.FeatureCollection('TIGER/2018/Counties')
    .filter(ee.Filter.eq('NAME', 'Mecklenburg'))
    .filter(ee.Filter.eq('STATEFP', '37'))
)

aoi_geom = AOI.geometry() # Optional: Geometry extraction for clipping

# Inspect AOI metadata
aoi_info = AOI.first().getInfo()
bbox = AOI.geometry().bounds().getInfo()['coordinates']
bbox_rounded = [[[round(coord, 4) for coord in point] for point in ring] for ring in bbox]
print("AOI successfully defined.")
print("County Name:", aoi_info['properties']['NAME'])
print("State FIPS:", aoi_info['properties']['STATEFP'])
print("Geometry Type:", aoi_info['geometry']['type'])
print("Approx. Bounding Box (4 decimals):", bbox_rounded)

AOI successfully defined.
County Name: Mecklenburg
State FIPS: 37
Geometry Type: Polygon
Approx. Bounding Box (4 decimals): [[[-81.0582, 35.0017], [-80.5503, 35.0017], [-80.5503, 35.5148], [-81.0582, 35.5148], [-81.0582, 35.0017]]]


# 2. Define Temporal Window
Select a single year for classification.

In [4]:
# ============================================================
# 2. Define Study Year
# ============================================================

YEAR = 1990

start_date = ee.Date.fromYMD(YEAR, 1, 1)
end_date = start_date.advance(1, 'year')

print(f"Study year: {YEAR}")

Study year: 1990


# 3. Select the Imagery Collection
Choose your collection from the metadata_collection notebook.

In [5]:
# ============================================================
# 3. Load Landsat 5 Collection
# ============================================================

collection_id = "LANDSAT/LT05/C02/T1_L2"

imagery_collection = (ee.ImageCollection(collection_id).filterBounds(aoi_geom).filterDate(start_date, end_date))

image_count = imagery_collection.size().getInfo() # Retrieve metadata for diagnostics

print(f"""
Collection Loaded Successfully
----------------------------------------
Dataset ID: {collection_id}
Study Year: {YEAR}
Spatial Resolution: 30 meters
Total Scenes Intersecting AOI: {image_count}
""")


Collection Loaded Successfully
----------------------------------------
Dataset ID: LANDSAT/LT05/C02/T1_L2
Study Year: 1990
Spatial Resolution: 30 meters
Total Scenes Intersecting AOI: 45



# 4. Preliminary processing

In [6]:
# ============================================================
# 4.1 Landsat 5 Quality Masking
# Cloud, Shadow, Snow & Radiometric Saturation Removal
# ============================================================

def mask_landsat5(image):
    """
    Robust masking function for LANDSAT/LT05/C02/T1_L2 (Surface Reflectance).

    This function removes:
        - Clouds (QA_PIXEL bit 3)
        - Cloud shadows (QA_PIXEL bit 4)
        - Snow/Ice (QA_PIXEL bit 5)
        - Radiometrically saturated pixels (QA_RADSAT band)

    It preserves essential temporal metadata for time-series analysis.
    """

    # ------------------------------------------------------------
    # 1. Extract Quality Assessment Bands
    # ------------------------------------------------------------
    
    # QA_PIXEL contains per-pixel condition flags encoded as bits
    qa_pixel = image.select('QA_PIXEL')
    
    # QA_RADSAT indicates radiometric saturation in spectral bands
    qa_radsat = image.select('QA_RADSAT')

    # ------------------------------------------------------------
    # 2. Define Bit Masks (Collection 2 Level-2 specification)
    # ------------------------------------------------------------

    # Bit positions for Landsat 5 Collection 2:
    # Bit 3  -> Cloud
    # Bit 4  -> Cloud Shadow
    # Bit 5  -> Snow / Ice
    
    cloud_bit_mask  = 1 << 3   # 00001000 (binary)
    shadow_bit_mask = 1 << 4   # 00010000 (binary)
    snow_bit_mask   = 1 << 5   # 00100000 (binary)

    # ------------------------------------------------------------
    # 3. Create Boolean Conditions for Each Flag
    # ------------------------------------------------------------

    # Keep pixel if cloud bit is NOT set (equals 0)
    no_cloud = qa_pixel.bitwiseAnd(cloud_bit_mask).eq(0)

    # Keep pixel if cloud shadow bit is NOT set
    no_shadow = qa_pixel.bitwiseAnd(shadow_bit_mask).eq(0)

    # Keep pixel if snow bit is NOT set
    no_snow = qa_pixel.bitwiseAnd(snow_bit_mask).eq(0)

    # ------------------------------------------------------------
    # 4. Combine QA Conditions
    # ------------------------------------------------------------

    # Pixel must satisfy ALL three conditions
    qa_combined_mask = no_cloud.And(no_shadow).And(no_snow)

    # ------------------------------------------------------------
    # 5. Radiometric Saturation Mask
    # ------------------------------------------------------------

    # QA_RADSAT == 0 means no band saturation
    no_saturation = qa_radsat.eq(0)

    # ------------------------------------------------------------
    # 6. Apply Combined Mask
    # ------------------------------------------------------------

    final_mask = qa_combined_mask.And(no_saturation)

    masked_image = image.updateMask(final_mask)

    # ------------------------------------------------------------
    # 7. Preserve Time Metadata (critical for temporal composites)
    # ------------------------------------------------------------

    return masked_image.copyProperties(
        image,
        ['system:time_start', 'system:index']
    )

# ============================================================
# 4.2 Apply Landsat Collection 2 Scaling Factors
# ============================================================

def apply_scaling_factors(image):
    """
    Applies scaling factors for Landsat 5 Collection 2 Level-2
    Surface Reflectance bands.

    Reflectance = (DN * 0.0000275) - 0.2
    """

    optical_bands = image.select([
        'SR_B1', 'SR_B2', 'SR_B3',
        'SR_B4', 'SR_B5', 'SR_B7'
    ])

    scaled_optical = optical_bands.multiply(0.0000275).add(-0.2)

    return image.addBands(scaled_optical, overwrite=True)

In [7]:
imagery_collection_clean = imagery_collection.map(mask_landsat5)
final_composite = imagery_collection_clean.map(apply_scaling_factors)

print("Scaling factors applied successfully.")

Scaling factors applied successfully.


# 5. Final Supervised Composite

In [8]:
# ============================================================
# 5. Build Annual Median Composite
# ============================================================

annual_composite = (final_composite.median().clip(aoi_geom))
print(f"Median composite for {YEAR} ready.")

Median composite for 1990 ready.


In [9]:
# ============================================================
# 5.1 Add Vegetation & Built-Up Indices (NDVI, NDBI, SAVI, EVI)
# ============================================================

def add_indices(image):
    ndvi = image.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')     # NDVI
    ndbi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDBI')     # NDBI

    # SAVI (L = 0.5 is standard)
    L = 0.5
    savi = image.expression(
        '((NIR - RED) / (NIR + RED + L)) * (1 + L)',
        {
            'NIR': image.select('SR_B4'),
            'RED': image.select('SR_B3'),
            'L': L
        }
    ).rename('SAVI')

    # EVI (standard coefficients)
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1))',
        {
            'NIR': image.select('SR_B4'),
            'RED': image.select('SR_B3'),
            'BLUE': image.select('SR_B1')
        }
    ).rename('EVI')

    return image.addBands([ndvi, ndbi, savi, evi])

annual_composite = add_indices(annual_composite)
print("NDVI, NDBI, SAVI, and EVI added.")

NDVI, NDBI, SAVI, and EVI added.


# 6. Visualize the Supervised Composite

In [10]:
# ============================================================
# 6. Visualization for Training Sample Selection
# ============================================================

supervised_map = geemap.Map()
supervised_map.centerObject(AOI)  # Auto zoom to AOI

# ------------------------------------------------------------
# True Color
# ------------------------------------------------------------
true_color_vis = {
    'bands': ['SR_B3', 'SR_B2', 'SR_B1'],
    'min': 0,
    'max': 0.3,
    'gamma': 1.2
}

# ------------------------------------------------------------
# False Color (Vegetation Highlight)
# ------------------------------------------------------------
false_color_vis = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0,
    'max': 0.4,
    'gamma': 1.1
}

# ------------------------------------------------------------
# NDVI Visualization
# ------------------------------------------------------------
ndvi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': ['brown', 'yellow', 'lightgreen', 'green', 'darkgreen']
}

# ------------------------------------------------------------
# SAVI Visualization
# ------------------------------------------------------------
savi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': ['brown', 'orange', 'yellow', 'lightgreen', 'darkgreen']
}

# ------------------------------------------------------------
# EVI Visualization
# ------------------------------------------------------------
evi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': ['purple', 'blue', 'cyan', 'lightgreen', 'darkgreen']
}

# ------------------------------------------------------------
# NDBI Visualization (Built-Up Highlight)
# ------------------------------------------------------------
ndbi_vis = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['darkgreen', 'white', 'orange', 'red']
}

In [17]:
# ------------------------------------------------------------
# Add Layers (toggle on/off during digitizing)
#------------------------------------------------------------
# supervised_map.addLayer(annual_composite, true_color_vis, f"True Color {YEAR}")
# supervised_map.addLayer(annual_composite, false_color_vis, f"False Color {YEAR}", False)
# supervised_map.addLayer(annual_composite.select('NDVI'), ndvi_vis, "NDVI", False)
# supervised_map.addLayer(annual_composite.select('SAVI'), savi_vis, "SAVI", False)
# supervised_map.addLayer(annual_composite.select('EVI'), evi_vis, "EVI", False)
# supervised_map.addLayer(annual_composite.select('NDBI'), ndbi_vis, "NDBI (Built-up)", False)
# supervised_map.addLayer(AOI.style(color='red', fillColor='00000000'), {}, "AOI Boundary")

In [12]:
# ============================================================
# Sanity Check
# ============================================================

num_layers = len(supervised_map.layers)
print("Number of layers in supervised_map:", num_layers)

# Optional: print layer names
print("\nLayer Names:")
for i, layer in enumerate(supervised_map.layers):
    print(f"{i + 1}. {layer.name}")

Number of layers in supervised_map: 2

Layer Names:
1. OpenStreetMap.Mapnik
2. Subdivision Centroids


# 7. Export to Google drive
With Earth Engine, you cannot export directly to your local computer disk.

Google Drive Export Destination: Google Drive → My Drive → GEE_visual_exports

Then you need to manually move the files into local drive

In [19]:
layer_configs = {
    "TrueColor": {
        "image": annual_composite,
        "vis": true_color_vis
    },
    "FalseColor": {
        "image": annual_composite,
        "vis": false_color_vis
    },
    "NDVI": {
        "image": annual_composite.select('NDVI'),
        "vis": ndvi_vis
    },
    "SAVI": {
        "image": annual_composite.select('SAVI'),
        "vis": savi_vis
    },
    "EVI": {
        "image": annual_composite.select('EVI'),
        "vis": evi_vis
    },
    "NDBI": {
        "image": annual_composite.select('NDBI'),
        "vis": ndbi_vis
    }
}

def export_visualized_layer(layer_name, image, vis_params, year):
    """
    Export image exactly as visualized in geemap.
    Converts styled layer to 8-bit RGB.
    """

    styled = image.visualize(**vis_params)

    task = ee.batch.Export.image.toDrive(
        image=styled,
        description=f"{layer_name}_{year}_Styled",
        folder="GEE_visual_exports",
        fileNamePrefix=f"{layer_name}_{year}_Styled",
        region=aoi_geom,
        scale=30,
        maxPixels=1e13
    )

    task.start()
    print(f"🚀 Export started: {layer_name}_{year}")

In [24]:
for name, config in layer_configs.items():
    export_visualized_layer(
        layer_name=name,
        image=config["image"],
        vis_params=config["vis"],
        year=YEAR
    )

🚀 Export started: TrueColor_1990
🚀 Export started: FalseColor_1990
🚀 Export started: NDVI_1990
🚀 Export started: SAVI_1990
🚀 Export started: EVI_1990
🚀 Export started: NDBI_1990


In [ ]:
# ============================================================
# 7.3 Checking the retrieving status
# ============================================================
for t in ee.batch.Task.list():
    print(t.status())

# 8. Training Sample Collection

In [13]:
# ============================================================
# 8.1 Drawing training samples
# ============================================================

supervised_map.add_draw_control() # Enable drawing tools

# ============================================================
# 8.2 NDVI Helper Masks
# ============================================================

ndvi = annual_composite.select('NDVI')

forest_candidate = ndvi.gt(0.6)
grass_candidate  = ndvi.gt(0.3).And(ndvi.lte(0.6))

supervised_map.addLayer(
    forest_candidate.selfMask(),
    {'palette':['darkgreen']},
    "Forest Candidate (NDVI > 0.6)",
    False
)

supervised_map.addLayer(
    grass_candidate.selfMask(),
    {'palette':['yellow']},
    "Grass Candidate (0.3–0.6)",
    False
)

# ============================================================
# 8.3 Export the samples and reset the map
# ============================================================

def export_current_rois(filename):
    fc = supervised_map.user_rois
    n = fc.size().getInfo()
    if n == 0:
        raise ValueError("No polygons drawn yet.")
    geemap.ee_export_geojson(fc, filename=filename)
    print(f"✅ Exported {n} polygons -> {filename}")

def reset_rois():
    # safest: rebuild the map object (geemap doesn't always expose a clean 'clear rois' API)
    global supervised_map
    supervised_map = geemap.Map()
    supervised_map.centerObject(AOI)
    supervised_map.add_draw_control()

    # re-add your helper layers (optional)
    supervised_map.addLayer(
        forest_candidate.selfMask(),
        {'palette':['darkgreen']},
        "Forest Candidate (NDVI > 0.6)",
        False
    )
    supervised_map.addLayer(
        grass_candidate.selfMask(),
        {'palette':['yellow']},
        "Grass Candidate (0.3–0.6)",
        False
    )
    return supervised_map

In [14]:
supervised_map

Map(center=[35.258247113131574, -80.80426010128677], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# --- Forest step ---
export_current_rois(
    f"../../../../Main/Landuse change/Erfan Vegetation/POIs/{YEAR}/forest_poi_{YEAR}_v1.geojson"
)

reset_rois() # --- Reset for next class ---

In [ ]:
# # --- Grass step ---
export_current_rois(
    f"../../../../Main/Landuse change/Erfan Vegetation/POIs/{YEAR}/grass_poi_{YEAR}_v1.geojson"
)